# Comparative Analysis of TensorBoard Runs
This notebook analyzes the logs for `run_008`, `run_009`, `run_010`, and `run_011`. It plots the custom losses and generates a summary table for the evaluation metrics.

In [12]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
from IPython.display import display, HTML

# Define runs to analyze
#run_names = ['run_008', 'run_009', 'run_010', 'run_011', 'run_012', 'run_013']
run_names = ['run_010', 'run_013']
runs_dir = '../runs'

all_data = []
for run in run_names:
    run_path = os.path.join(runs_dir, run)
    if not os.path.exists(run_path):
        print(f'Run directory {run_path} does not exist')
        continue
        
    # Load all event files within the run directory (including subdirectories for eval)
    event_files = sorted(glob.glob(os.path.join(run_path, '**', 'events.out.tfevents.*'), recursive=True))
    print(f'Found {len(event_files)} event files for {run}')
    
    for ev_file in event_files:
        # Size guidance to ensure we load enough scalars. 0 means load all.
        size_guidance = {'scalars': 0}
        ea = event_accumulator.EventAccumulator(ev_file, size_guidance=size_guidance)
        ea.Reload()
        
        if 'scalars' in ea.Tags():
            tags = ea.Tags()['scalars']
            for tag in tags:
                for ev in ea.Scalars(tag):
                    all_data.append({'run': run, 'tag': tag, 'step': ev.step, 'value': ev.value})

if all_data:
    df = pd.DataFrame(all_data)
    # Remove duplicates by taking the last value for a given (run, tag, step)
    df = df.groupby(['run', 'tag', 'step']).last().reset_index()
    print('\nData successfully loaded and aggregated.')
else:
    print('\nNo data found in the specified runs.')
    df = pd.DataFrame(columns=['run', 'tag', 'step', 'value'])

Found 3 event files for run_010
Found 2 event files for run_013

Data successfully loaded and aggregated.


## Smoothed Losses with Confidence Areas
Plotting the custom losses (`train/cls_loss`, `train/reg_loss`) and overall loss if available, applying a rolling window for smoothing and plotting to show confidence variance.

In [13]:
import plotly.graph_objects as go

def plot_smoothed_with_conf(df, metric, window=100):
    metric_df = df[df['tag'] == metric]
    if metric_df.empty:
        return
        
    fig = go.Figure()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    for idx, run in enumerate(run_names):
        run_data = metric_df[metric_df['run'] == run].sort_values('step')
        if run_data.empty:
            continue
            
        run_data = run_data.set_index('step')['value']
        
        # Calculate moving average and standard deviation
        rolling_mean = run_data.rolling(window=window, min_periods=max(1, window//10)).mean()
        rolling_std = run_data.rolling(window=window, min_periods=max(1, window//10)).std().fillna(0)
        
        color = colors[idx % len(colors)]
        
        # Plot the confidence area (variance) around the smooth line
        upper_bound = rolling_mean + rolling_std
        lower_bound = rolling_mean - rolling_std
        
        fig.add_trace(go.Scatter(
            x=run_data.index.tolist() + run_data.index.tolist()[::-1],
            y=upper_bound.tolist() + lower_bound.tolist()[::-1],
            fill='toself',
            fillcolor=color,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False,
            name=f'{run} bounds',
            opacity=0.2
        ))
        
        # Plot the smoothed line
        fig.add_trace(go.Scatter(
            x=run_data.index,
            y=rolling_mean,
            mode='lines',
            line=dict(color=color, width=2),
            name=f"{run} (smoothed)"
        ))
        
        # Plot the faint original data points
        fig.add_trace(go.Scatter(
            x=run_data.index,
            y=run_data,
            mode='lines',
            line=dict(color=color, width=1),
            opacity=0.3,
            showlegend=False,
            name=f"{run} (raw)"
        ))
        
    fig.update_layout(
        title=f'{metric} (Smoothed window={window})',
        xaxis_title='Steps',
        yaxis_title=metric,
        hovermode="x unified",
        template='plotly_white',
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
    )
    fig.show()

# Identify and plot the custom training losses
target_losses = ['train/train/cls_loss', 'train/train/reg_loss', 'train/loss']
available_losses = [tag for tag in target_losses if tag in df['tag'].unique()]

for l in available_losses:
    plot_smoothed_with_conf(df, l, window=50) # Use a window of 50 for smoothing

## Evaluation Metrics Summary Table by Hierarchy
Here we extract all custom evaluation metrics (e.g., `mAP` metrics written by `eval.py`) and summarize the best achieved values per run, highlighting the most performant model, grouped by the object hierarchy.

In [14]:
# Extract Custom evaluation metrics
eval_tags = [t for t in df['tag'].unique() if 'mAP' in t or 'eval' in t]
eval_tags = [t for t in eval_tags if t not in ['eval/loss']]

if not eval_tags:
    print("No evaluation metrics (mAP) found in the logs.")
else:
    # Compute best metrics per run
    best_metrics = []
    for tag in eval_tags:
        for run in run_names:
            run_data = df[(df['tag'] == tag) & (df['run'] == run)]
            if not run_data.empty:
                max_val = run_data['value'].max()
                best_metrics.append({'Metric': tag, 'Run': run, 'Best Value': max_val})
                
    best_df = pd.DataFrame(best_metrics)

    # Load hierarchy
    import json
    import plotly.express as px

    hierarchy_path = '../hierarchy.json'
    hierarchy = {}
    if os.path.exists(hierarchy_path):
        with open(hierarchy_path, 'r') as f:
            hierarchy = json.load(f)

    groups = {'Global Metrics': [t for t in eval_tags if t.startswith('mAP/') or not t.startswith('mAP_Class/')]} 
    
    for group_name, classes in hierarchy.items():
        # The tags in the logs are like 'mAP_Class/note'
        group_tags = [f'mAP_Class/{cls}' for cls in classes if f'mAP_Class/{cls}' in eval_tags]
        if group_tags:
            groups[group_name] = group_tags
            
    # Ensure any ungrouped mAP_Class tags are not lost
    grouped_tags = set(sum(groups.values(), []))
    ungrouped = [t for t in eval_tags if t not in grouped_tags]
    if ungrouped:
        groups['Other Classes'] = ungrouped

    # Generate tables and plots per group
    for group_name, tags in groups.items():
        group_df = best_df[best_df['Metric'].isin(tags)].copy()
        if group_df.empty:
            continue
            
        # 1. Summary Table
        summary_df = group_df.pivot(index='Metric', columns='Run', values='Best Value')
        for r in run_names:
            if r not in summary_df.columns:
                summary_df[r] = np.nan
                
        def highlight_max(s):
            is_max = s == s.max()
            return ['font-weight: bold; background-color: #d4edda; color: black' if v else '' for v in is_max]
            
        styled_df = (
            summary_df.style
            .apply(highlight_max, axis=1)
            .format("{:.4f}", na_rep="-")
            .set_caption(f"{group_name.replace('_', ' ')} - Maximum Performance by Model")
            .set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'left'), ('font-size', '14px')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px')]},
                {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
            ])
        )
        display(HTML(styled_df.to_html()))
        
        # 2. Plotly Plot
        # Strip the prefix for cleaner legends and ticks
        group_df['Metric Name'] = group_df['Metric'].str.replace('mAP_Class/', '').str.replace('mAP/', '')
        
        fig = px.line(group_df, x='Run', y='Best Value', color='Metric Name', symbol='Metric Name',
                      markers=True, title=f'{group_name.replace("_", " ")} - Plot Comparison',
                      template='plotly_white')
        # Add some padding to Y axis so markers are not cut off
        fig.update_yaxes(range=[max(0, group_df['Best Value'].min()-0.05), min(1.05, group_df['Best Value'].max()+0.05)])
        fig.update_traces(marker=dict(size=12), line=dict(width=2))
        fig.update_layout(xaxis_title='Run', yaxis_title='Best mAP Score', hovermode='x unified')
        fig.show()


Run,run_010,run_013
Metric,,
mAP/IoU_0.50,0.4805,0.4415
mAP/IoU_0.50_0.95,0.3508,0.3207
mAP/IoU_0.75,0.3905,0.3469


Run,run_010,run_013
Metric,,
mAP_Class/chord,0.5140,0.5057
mAP_Class/mRest,0.2138,0.6705
mAP_Class/note,0.4482,0.4402
mAP_Class/notehead,0.5198,0.5007
mAP_Class/rest,0.3778,0.2558


Run,run_010,run_013
Metric,,
mAP_Class/barLine,0.1147,0.1186
mAP_Class/clef,0.9236,0.9126
mAP_Class/keyAccid,0.7606,0.3191
mAP_Class/keySig,0.5561,0.5363
mAP_Class/ledgerLines,0.0068,0.0046
mAP_Class/measure,0.4684,0.3949
mAP_Class/meterSig,0.6894,0.3862
mAP_Class/staff,0.6440,0.6694


Run,run_010,run_013
Metric,,
mAP_Class/beam,0.6039,0.5484
mAP_Class/flag,0.4914,0.5653
mAP_Class/slur,0.1903,0.2164
mAP_Class/stem,0.4423,0.3351
mAP_Class/tie,0.2619,0.2560
mAP_Class/tuplet,0.0006,0.0000
mAP_Class/tupletNum,0.0002,0.0358


Run,run_010,run_013
Metric,,
mAP_Class/arpeg,0.0000,0.0000
mAP_Class/artic,0.0002,0.0000
mAP_Class/bTrem,-,-1.0000
mAP_Class/fermata,0.3045,0.2606
mAP_Class/mordent,0.0000,0.0000
mAP_Class/trill,0.0000,0.0026


...

Run,run_010,run_013
Metric,,
mAP_Class/dir,0.0695,0.0160
mAP_Class/dynam,0.4768,0.3076
mAP_Class/tempo,0.5863,0.3913


Run,run_010,run_013
Metric,,
mAP_Class/label,0.1663,0.1930
mAP_Class/labelAbbr,0.6429,0.6261
mAP_Class/mNum,-1.0000,-1.0000
mAP_Class/pgHead,0.8089,0.7509
mAP_Class/reh,0.0000,0.0000


Run,run_010,run_013
Metric,,
mAP_Class/ending,-1.0000,-1.0000
mAP_Class/grpSym,0.8369,0.7780
mAP_Class/repeatMark,0.3762,0.0000
mAP_Class/voltaBracket,0.2663,0.0066


Run,run_010,run_013
Metric,,
mAP_Class/fing,0.0000,0.1687
mAP_Class/syl,0.2830,0.4284
mAP_Class/tabDurSym,0.0000,0.0000
mAP_Class/tabGrp,0.0212,0.0000
mAP_Class/verse,0.4118,0.2483


Run,run_010,run_013
Metric,,
mAP_Class/autogenerated,0.0000,0.4690
mAP_Class/layer,0.3798,0.3063
mAP_Class/page-margin,0.7676,0.7751
mAP_Class/svg,0.9505,0.9821
mAP_Class/system,0.7026,0.6861


In [15]:
summary_df.to_markdown()

'| Metric                  |   run_010 |   run_013 |\n|:------------------------|----------:|----------:|\n| mAP_Class/autogenerated |  0        |  0.468998 |\n| mAP_Class/layer         |  0.379798 |  0.306274 |\n| mAP_Class/page-margin   |  0.767614 |  0.77506  |\n| mAP_Class/svg           |  0.950495 |  0.982123 |\n| mAP_Class/system        |  0.702558 |  0.686097 |'

In [17]:
df

,run,tag,step,value
0,run_010,mAP/IoU_0.50,59880,0.480502
1,run_010,mAP/IoU_0.50_0.95,59880,0.350798
2,run_010,mAP/IoU_0.75,59880,0.390485
3,run_010,mAP_Class/accid,59880,0.569851
4,run_010,mAP_Class/arpeg,59880,0.000000
...,...,...,...,...
10938,run_013,train/train/reg_loss,30240,0.364206
10939,run_013,train/train_loss,30240,3.158228
10940,run_013,train/train_runtime,30240,1873.607056
10941,run_013,train/train_samples_per_second,30240,115.573997
